# Evaluate a safety classifier on MindGuard-testset

Score any classifier (Qwen3Guard-style fine-tuned model) served via a vLLM
OpenAI-compatible endpoint and report **AUROC**, **FPR@90%TPR**, **FPR@95%TPR**,
and a 3x3 confusion matrix.

## Setup
Serve the classifier with vLLM:
```bash
vllm serve <classifier-checkpoint> --port 8000
```

## Configuration

In [ ]:
MODEL_NAME = ""   # e.g. "hosted_vllm/MindGuard-4B"
API_BASE   = ""   # e.g. "http://localhost:8000/v1"
MAX_WORKERS = 32


## Imports

In [ ]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import litellm
import numpy as np
from datasets import load_dataset
from sklearn.metrics import roc_curve, auc, confusion_matrix
from tqdm.auto import tqdm

## Load the test set


In [ ]:
ds = load_dataset("swordhealth/MindGuard-testset", split="test")
print(f"N = {len(ds)}")
print("Label distribution:")
for label, count in Counter(ds["label"]).most_common():
    print(f"  {label:<25} {count}")

## Classify one conversation


In [ ]:
def parse_label(text):
    """Argmax label from MindGuard text output."""
    t = text.lower()
    if "safety: unsafe" in t:
        if "s2" in t:
            return "unsafe_harm_to_others"
        return "unsafe_self_harm_risk"  # default S1 / unspecified unsafe
    return "safe"


def classify(messages):
    """Return (label, prob_unsafe, raw_text) for a list of role/content messages."""
    resp = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
        api_base=API_BASE,
        temperature=0,
        logprobs=True,
        top_logprobs=5,
    )
    text  = resp.choices[0].message["content"]
    lp    = resp.choices[0].logprobs.content

    token = lp[2].token
    logp  = lp[2].logprob
    if token == " Unsafe":
        prob_unsafe = float(np.exp(logp))
    elif token == " Safe":
        prob_unsafe = 1.0 - float(np.exp(logp))
    else:
        prob_unsafe = 0.0
        for alt in lp[2].top_logprobs:
            if alt.token == " Unsafe":
                prob_unsafe = float(np.exp(alt.logprob)); break
            if alt.token == " Safe":
                prob_unsafe = 1.0 - float(np.exp(alt.logprob)); break
    return parse_label(text), prob_unsafe, text


# Quick sanity check on the first row
label, p, text = classify(ds[0]["prompt"])
print("GT:        ", ds[0]["label"])
print("Pred label:", label)
print("Output:    ", text.strip().splitlines()[0])
print("P(unsafe): ", round(p, 4))

## Score the full test set in parallel

In [ ]:
def safe_classify(idx_messages):
    idx, messages = idx_messages
    try:
        label, p, _ = classify(messages)
        return idx, label, p, None
    except Exception as e:
        return idx, None, None, str(e)


indexed = list(enumerate(ds["prompt"]))
labels  = [None] * len(ds)
scores  = [None] * len(ds)
errors  = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for idx, label, p, err in tqdm(ex.map(safe_classify, indexed), total=len(indexed)):
        if err is not None:
            errors += 1
            continue
        labels[idx] = label
        scores[idx] = p

print(f"Failed: {errors}/{len(ds)}")

## Threshold-free metrics


In [ ]:
y_true   = np.array([0 if l == "safe" else 1 for l in ds["label"]])
y_scores = np.array([s if s is not None else 0.0 for s in scores])

fpr, tpr, _ = roc_curve(y_true, y_scores)
auroc       = auc(fpr, tpr)

def fpr_at_tpr(target):
    idx = np.where(tpr >= target)[0]
    return float(fpr[idx[0]]) if len(idx) else float("nan")

print(f"AUROC      : {auroc:.3f}")
print(f"FPR@90%TPR : {fpr_at_tpr(0.90):.3f}")
print(f"FPR@95%TPR : {fpr_at_tpr(0.95):.3f}")

## Confusion matrix (3x3, argmax)


In [ ]:
CLASSES = ["safe", "unsafe_self_harm_risk", "unsafe_harm_to_others"]
SHORT   = {"safe": "Safe", "unsafe_self_harm_risk": "SAR", "unsafe_harm_to_others": "HTO"}

y_true_mc = list(ds["label"])
y_pred_mc = [l if l is not None else "safe" for l in labels]

cm = confusion_matrix(y_true_mc, y_pred_mc, labels=CLASSES)

header = f"{'':>8}" + " ".join(f"{SHORT[c]:>6}" for c in CLASSES)
print("Predicted")
print(header)
for i, c in enumerate(CLASSES):
    row = f"{SHORT[c]:>8}" + " ".join(f"{cm[i,j]:>6d}" for j in range(len(CLASSES)))
    print(row)
print("\n(rows = ground truth, columns = predicted)")